# OTFS调制原理

**OTFS Modulation Principles**

---

本notebook是OTFS通信原理教程的最后一篇，汇总了所有前面的概念：
- Fourier变换（信号表示）
- 延迟-多普勒域（信道表示）
- 多径信道与多普勒效应（信道特性）
- QAM调制、OFDM（调制技术）
- MIMO（多天线技术）
- Zak变换 / SFFT（OTFS核心数学工具）

## 1. 目标

通过本notebook的学习，你将：

- **理解OTFS调制的核心原理**：为什么在延迟-多普勒域放置QAM符号
- **掌握OTFS调制解调全流程**：QAM → SFFT → OFDM → 信道 → OFDM解调 → ISFFT → QAM解映射
- **理解时频域网格与延迟-多普勒域网格的变换关系**
- **对比OTFS与OFDM的性能差异**，特别是高移动性场景下

## 2. OTFS调制原理

### 2.1 核心思想

OTFS（正交时频空）的核心创新是：**将QAM信息符号放置在延迟-多普勒域，而非传统的时频域**。

这种设计的物理动机来自无线信道的特性：

| 域 | 数据符号位置 | 信道响应 | 均衡 |
|------|------------|----------|------|
| **OFDM时频域** | $(t, f)$ 时频网格 | $H(t,f)$ 时变 | 每子载波多抽头 |
| **OTFS延迟-多普勒域** | $(\tau, \nu)$ 延迟-多普勒网格 | $h(\tau,\nu)$ 时不变 | 单抽头 |

### 2.2 OTFS调制流程

OTFS将QAM符号从延迟-多普勒域变换到时域发射：

```
QAM符号 X(τ, ν)   ← 延迟-多普勒域（信息放置位置）
       ↓
   SFFT变换
       ↓
时频域信号 X(t, f)
       ↓
   IFFT + CP
       ↓
   时域信号 x(t)  →  射频发射
```

### 2.3 关键洞察：时不变信道

在延迟-多普勒域，**信道对所有QAM符号的作用是相同的**。即：

$$Y(\tau, \nu) = H(\tau, \nu) \cdot X(\tau, \nu)$$

其中 $H(\tau, \nu)$ 是信道在延迟-多普勒域的响应，**与时间无关**！

这意味着每个QAM符号只需要简单的复数乘法即可完成均衡。

## 3. 代码演示：OTFS调制解调全流程

下面我们实现一个完整的OTFS系统，包含：

1. **发送端**：QAM符号 → 延迟-多普勒网格 → SFFT → 时频域 → IFFT+CP → 时域
2. **信道**：多径 + 多普勒
3. **接收端**：去除CP+FFT → ISFFT → 延迟-多普勒域 → 均衡 → QAM解映射

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Set up Chinese font support
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("OTFS系统演示包加载成功")

### 3.1 系统参数设置

In [ ]:
# OTFS系统参数
N = 32   # 时间维度（OFDM符号数）/ 多普勒采样数
M = 32   # 频率维度（子载波数）/ 延迟采样数
cp_len = 8  # 循环前缀长度

# QAM参数
qam_order = 16  # 16-QAM

# 信道参数
num_paths = 3  # 多径数量
max_delay = 5  # 最大延迟（采样点）
max_doppler = 3  # 最大多普勒偏移

print(f"OTFS网格大小: {N} × {M}")
print(f"总QAM符号数: {N * M}")
print(f"循环前缀长度: {cp_len}")
print(f"子载波间隔: 1/{M}")

### 3.2 SFFT（对称有限傅里叶变换）函数定义

In [ ]:
def sfft(x, N, M):
    """
    对称有限傅里叶变换（SFFT）= Zak变换
    
    将时域/频域信号变换到延迟-多普勒域
    
    参数:
        x: 输入信号，shape (N, M) - N个时间采样，M个频率采样
        N: 时间维度
        M: 频率维度
    
    返回:
        X_dd: 延迟-多普勒域信号，shape (M, N)
    
    SFFT = FFT行 + IFFT列
    """
    # 第一步：FFT沿行方向（时间→频率）
    X = np.fft.fft(x, axis=1)
    # 第二步：IFFT沿列方向（频率→延迟）
    X_dd = np.fft.ifft(X, axis=0)
    return X_dd

def isfft(X_dd, N, M):
    """
    逆SFFT（ISFFT）
    
    将延迟-多普勒域信号变换到时域/频域
    
    参数:
        X_dd: 延迟-多普勒域信号，shape (M, N)
        N: 时间维度
        M: 频率维度
    
    返回:
        x: 时域/频域信号，shape (N, M)
    
    ISFFT = FFT列 + IFFT行
    """
    # 第一步：FFT沿列方向
    X = np.fft.fft(X_dd, axis=0)
    # 第二步：IFFT沿行方向
    x = np.fft.ifft(X, axis=1)
    return x

print("SFFT/ISFFT函数定义完成")

### 3.3 创建QAM星座图和映射函数

In [ ]:
def create_qam_constellation(order):
    """创建QAM星座图"""
    if order == 16:
        levels = np.array([-3, -1, 1, 3]) / 3.0
        constellation = []
        for i in levels:
            for j in levels:
                constellation.append(i + 1j * j)
        return np.array(constellation), 4
    elif order == 4:
        return np.array([1+0j, 0+1j, -1+0j, 0-1j]), 2
    else:
        raise ValueError(f"Unsupported QAM order: {order}")

def qam_modulate(bits, constellation):
    """将比特流映射到QAM星座点"""
    bits_per_symbol = int(np.log2(len(constellation)))
    symbols = []
    for i in range(0, len(bits), bits_per_symbol):
        if i + bits_per_symbol <= len(bits):
            symbol_bits = bits[i:i+bits_per_symbol]
            symbol_idx = int(''.join(map(str, symbol_bits)), 2)
            symbols.append(constellation[symbol_idx])
    return np.array(symbols)

def qam_demodulate(symbols, constellation):
    """将QAM符号解映射回比特"""
    bits_per_symbol = int(np.log2(len(constellation)))
    recovered_bits = []
    for symbol in symbols:
        distances = np.abs(constellation - symbol)
        nearest_idx = np.argmin(distances)
        bit_str = format(nearest_idx, f'0{bits_per_symbol}b')
        recovered_bits.extend([int(b) for b in bit_str])
    return np.array(recovered_bits)

# 创建16-QAM星座
qam_constellation, qam_bits = create_qam_constellation(qam_order)
print(f"QAM阶数: {qam_order}")
print(f"每个符号携带 {qam_bits} 比特")
print(f"星座点数量: {len(qam_constellation)}")

### 3.4 OTFS发送端：QAM → 延迟-多普勒域 → SFFT → OFDM调制

In [ ]:
# 生成随机比特流并调制为QAM符号
np.random.seed(42)
total_bits = N * M * qam_bits
tx_bits = np.random.randint(0, 2, total_bits)

# 调制为QAM符号
qam_symbols = qam_modulate(tx_bits, qam_constellation)
print(f"生成QAM符号数: {len(qam_symbols)}")

# 将QAM符号放置在延迟-多普勒网格上
# X_dd[delay, doppler] - shape (M, N)
X_dd = qam_symbols.reshape(M, N)
print(f"延迟-多普勒网格形状: {X_dd.shape}")
print(f"  - 延迟维度（行）: {M}")
print(f"  - 多普勒维度（列）: {N}")

In [ ]:
# 可视化：延迟-多普勒域的QAM符号
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 幅度
ax1 = axes[0]
im1 = ax1.imshow(np.abs(X_dd), aspect='auto', origin='lower',
                  extent=[0, N, 0, M], cmap='Blues', interpolation='nearest')
plt.colorbar(im1, ax=ax1, label='|X(τ, ν)|')
ax1.set_xlabel('多普勒索引 (ν)', fontsize=11)
ax1.set_ylabel('延迟索引 (τ)', fontsize=11)
ax1.set_title('OTFS发送：延迟-多普勒域QAM符号（幅度）', fontsize=12)

# 相位
ax2 = axes[1]
im2 = ax2.imshow(np.angle(X_dd), aspect='auto', origin='lower',
                  extent=[0, N, 0, M], cmap='hsv', interpolation='nearest',
                  vmin=-np.pi, vmax=np.pi)
plt.colorbar(im2, ax=ax2, label='相位 (rad)', shrink=0.8)
ax2.set_xlabel('多普勒索引 (ν)', fontsize=11)
ax2.set_ylabel('延迟索引 (τ)', fontsize=11)
ax2.set_title('OTFS发送：延迟-多普勒域QAM符号（相位）', fontsize=12)

plt.tight_layout()
plt.show()

print("观察：每个格点(delay, doppler)上放置一个QAM符号")
print("这是OTFS的核心：信息符号在延迟-多普勒域表示")

In [ ]:
# SFFT变换：延迟-多普勒域 → 时频域
# X_tf[time, freq] - shape (N, M)
X_tf = isfft(X_dd, N, M)
print(f"SFFT变换完成")
print(f"时频域网格形状: {X_tf.shape}")
print(f"  - 时间维度（行）: {N}")
print(f"  - 频率维度（列）: {M}")

In [ ]:
# 可视化：SFFT后的时频域信号
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 时频域幅度
ax1 = axes[0]
im1 = ax1.imshow(np.abs(X_tf), aspect='auto', origin='lower',
                  extent=[0, M, 0, N], cmap='Greens', interpolation='nearest')
plt.colorbar(im1, ax=ax1, label='|X(t, f)|')
ax1.set_xlabel('频率索引 (f)', fontsize=11)
ax1.set_ylabel('时间索引 (t)', fontsize=11)
ax1.set_title('SFFT后：时频域信号（幅度）', fontsize=12)

# 时频域相位
ax2 = axes[1]
im2 = ax2.imshow(np.angle(X_tf), aspect='auto', origin='lower',
                  extent=[0, M, 0, N], cmap='hsv', interpolation='nearest',
                  vmin=-np.pi, vmax=np.pi)
plt.colorbar(im2, ax=ax2, label='相位 (rad)', shrink=0.8)
ax2.set_xlabel('频率索引 (f)', fontsize=11)
ax2.set_ylabel('时间索引 (t)', fontsize=11)
ax2.set_title('SFFT后：时频域信号（相位）', fontsize=12)

plt.tight_layout()
plt.show()

print("观察：SFFT后，稀疏的延迟-多普勒域信号变成时频域的密集分布")
print("但接收端通过ISFFT可以完美恢复原始稀疏信号")

In [ ]:
# OFDM调制：时频域 → 时域（含CP）
def ofdm_modulate(tf_grid, n_fft, cp_len):
    """
    OFDM调制：将时频域信号转换为时域OFDM符号
    
    参数:
        tf_grid: shape (N_time, M_freq) - 时频域信号
        n_fft: FFT大小
        cp_len: 循环前缀长度
    
    返回:
        ofdm_symbols: shape (N_time, n_fft + cp_len) - 含CP的时域OFDM符号
    """
    num_symbols = tf_grid.shape[0]
    ofdm_symbols = []
    
    for i in range(num_symbols):
        # IFFT：将频域信号转换到时域
        time_domain = np.fft.ifft(tf_grid[i], n=n_fft)
        
        # 添加循环前缀
        cp = time_domain[-cp_len:]
        ofdm_with_cp = np.concatenate([cp, time_domain])
        ofdm_symbols.append(ofdm_with_cp)
    
    return np.array(ofdm_symbols)

# 执行OFDM调制
tx_ofdm_symbols = ofdm_modulate(X_tf, M, cp_len)
print(f"OFDM调制完成")
print(f"每个OFDM符号长度: {M + cp_len} (数据: {M} + CP: {cp_len})")
print(f"总OFDM符号形状: {tx_ofdm_symbols.shape}")

In [ ]:
# 可视化：OFDM时域信号
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 第一个OFDM符号的时域波形
ax1 = axes[0, 0]
ofdm_idx = 0
time_axis = np.arange(len(tx_ofdm_symbols[ofdm_idx]))
ax1.plot(time_axis, tx_ofdm_symbols[ofdm_idx].real, 'b-', label='实部', alpha=0.7)
ax1.plot(time_axis, tx_ofdm_symbols[ofdm_idx].imag, 'r-', label='虚部', alpha=0.7)
ax1.axvline(x=cp_len, color='g', linestyle='--', label='CP结束')
ax1.set_xlabel('采样点', fontsize=11)
ax1.set_ylabel('幅度', fontsize=11)
ax1.set_title(f'OFDM符号#{ofdm_idx}时域波形（实部/虚部）', fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)

# 第一个OFDM符号的频谱
ax2 = axes[0, 1]
# 去除CP后做FFT
freq_response = np.fft.fft(tx_ofdm_symbols[ofdm_idx][cp_len:], n=M)
ax2.stem(np.arange(M), np.abs(freq_response), linefmt='b-', markerfmt='bo', basefmt='k-')
ax2.set_xlabel('子载波索引', fontsize=11)
ax2.set_ylabel('幅度', fontsize=11)
ax2.set_title('OFDM符号频谱', fontsize=12)
ax2.grid(True, alpha=0.3)

# 多个OFDM符号的时域信号（拼接）
ax3 = axes[1, 0]
tx_time = tx_ofdm_symbols.flatten()
time_full = np.arange(len(tx_time))
ax3.plot(time_full, tx_time.real, 'b-', alpha=0.5, linewidth=0.5)
ax3.set_xlabel('采样点', fontsize=11)
ax3.set_ylabel('幅度', fontsize=11)
ax3.set_title(f'连续{N}个OFDM符号（实部）', fontsize=12)
ax3.grid(True, alpha=0.3)

# 星座图（发送）
ax4 = axes[1, 1]
qam_flat = X_dd.flatten()
ax4.scatter(qam_flat.real, qam_flat.imag, c='blue', s=20, alpha=0.5, edgecolors='black', linewidths=0.3)
ax4.axhline(y=0, color='k', linewidth=0.5)
ax4.axvline(x=0, color='k', linewidth=0.5)
ax4.set_xlabel('In-Phase (I)', fontsize=11)
ax4.set_ylabel('Quadrature (Q)', fontsize=11)
ax4.set_title('发送QAM星座图', fontsize=12)
ax4.grid(True, alpha=0.3)
ax4.set_xlim(-1.5, 1.5)
ax4.set_ylim(-1.5, 1.5)
ax4.set_aspect('equal')

plt.tight_layout()
plt.show()

### 3.5 多径多普勒信道

In [ ]:
# 创建多径多普勒信道
def create_delay_doppler_channel(num_paths, max_delay, max_doppler, N, M):
    """
    创建延迟-多普勒域信道
    
    返回:
        h_dd: 延迟-多普勒信道，shape (max_delay+1, 2*max_doppler+1)
        channel_taps: (delay_idx, doppler_idx, amplitude) 列表
    """
    np.random.seed(100)
    
    # 信道在延迟-多普勒域是稀疏的
    h_dd = np.zeros((max_delay + 1, 2 * max_doppler + 1), dtype=complex)
    
    channel_taps = []
    
    # 路径1：主径 (delay=0, doppler=0)
    h_dd[0, max_doppler] = 1.0 + 0j
    channel_taps.append((0, 0, 1.0 + 0j))
    
    # 其他随机路径
    for _ in range(num_paths - 1):
        delay_idx = np.random.randint(1, max_delay + 1)
        doppler_idx = np.random.randint(0, 2 * max_doppler + 1) - max_doppler
        amplitude = (np.random.randn() + 1j * np.random.randn()) * 0.5
        h_dd[delay_idx, doppler_idx + max_doppler] = amplitude
        channel_taps.append((delay_idx, doppler_idx, amplitude))
    
    return h_dd, channel_taps

def apply_channel_to_otfs(tx_signal, channel_taps, max_delay, max_doppler, cp_len, M):
    """
    将多径多普勒信道应用于OTFS信号
    
    参数:
        tx_signal: 含CP的时域OFDM符号
        channel_taps: (delay_idx, doppler_idx, amplitude)
        max_delay: 最大延迟
        max_doppler: 最大多普勒
        cp_len: CP长度
        M: FFT大小
    
    返回:
        rx_signal: 接收信号
    """
    rx_signal = np.zeros_like(tx_signal, dtype=complex)
    
    for delay_idx, doppler_idx, amplitude in channel_taps:
        # 对每条路径进行延迟和多普勒偏移
        for sym_idx in range(len(tx_signal)):
            # 该路径在该符号上的复增益（考虑多普勒）
            doppler_phase = np.exp(1j * 2 * np.pi * doppler_idx * sym_idx / N)
            gain = amplitude * doppler_phase
            
            # 延迟应用
            start = cp_len + delay_idx
            end = start + M
            if end <= len(tx_signal[sym_idx]):
                rx_signal[sym_idx, start:end] += tx_signal[sym_idx, cp_len:cp_len+M] * gain
    
    # 添加噪声
    rx_signal += 0.01 * (np.random.randn(*rx_signal.shape) + 1j * np.random.randn(*rx_signal.shape))
    
    return rx_signal

# 创建信道
h_dd, channel_taps = create_delay_doppler_channel(num_paths, max_delay, max_doppler, N, M)
print("延迟-多普勒信道创建完成")
print(f"信道形状: {h_dd.shape}")
print("路径信息:")
for delay_idx, doppler_idx, amp in channel_taps:
    print(f"  延迟={delay_idx}, 多普勒={doppler_idx}, 幅度={np.abs(amp):.2f}, 相位={np.angle(amp):.2f}")

In [ ]:
# 可视化延迟-多普勒信道
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 幅度
ax1 = axes[0]
im1 = ax1.imshow(np.abs(h_dd), aspect='auto', origin='lower',
                  extent=[-max_doppler, max_doppler, 0, max_delay+1],
                  cmap='hot', interpolation='nearest')
plt.colorbar(im1, ax=ax1, label='|h(τ, ν)|')
ax1.set_xlabel('多普勒索引 (ν)', fontsize=11)
ax1.set_ylabel('延迟索引 (τ)', fontsize=11)
ax1.set_title('延迟-多普勒信道幅度 |h(τ, ν)|', fontsize=12)

# 相位
ax2 = axes[1]
im2 = ax2.imshow(np.angle(h_dd), aspect='auto', origin='lower',
                  extent=[-max_doppler, max_doppler, 0, max_delay+1],
                  cmap='hsv', interpolation='nearest', vmin=-np.pi, vmax=np.pi)
plt.colorbar(im2, ax=ax2, label='相位 (rad)', shrink=0.8)
ax2.set_xlabel('多普勒索引 (ν)', fontsize=11)
ax2.set_ylabel('延迟索引 (τ)', fontsize=11)
ax2.set_title('延迟-多普勒信道相位 ∠h(τ, ν)', fontsize=12)

plt.tight_layout()
plt.show()

print("观察：信道在延迟-多普勒域是稀疏的 - 只有几个非零点")
print("这正是OTFS利用的特性：稀疏信道易于估计")

In [ ]:
# 通过信道
rx_ofdm_symbols = apply_channel_to_otfs(tx_ofdm_symbols, channel_taps, max_delay, max_doppler, cp_len, M)
print(f"信号通过信道")
print(f"发射信号形状: {tx_ofdm_symbols.shape}")
print(f"接收信号形状: {rx_ofdm_symbols.shape}")

### 3.6 OTFS接收端：OFDM解调 + ISFFT + 均衡 + QAM解映射

In [ ]:
# OFDM解调：时域 → 时频域
def ofdm_demodulate(rx_ofdm_symbols, n_fft, cp_len):
    """
    OFDM解调：将时域OFDM符号转换为频域
    
    参数:
        rx_ofdm_symbols: shape (N_time, n_fft + cp_len) - 含CP的接收OFDM符号
        n_fft: FFT大小
        cp_len: 循环前缀长度
    
    返回:
        tf_grid: shape (N_time, n_fft) - 时频域信号
    """
    num_symbols = rx_ofdm_symbols.shape[0]
    tf_grid = []
    
    for i in range(num_symbols):
        # 去除CP
        symbol = rx_ofdm_symbols[i, cp_len:cp_len + n_fft]
        
        # FFT：时域 → 频域
        freq_domain = np.fft.fft(symbol, n=n_fft)
        tf_grid.append(freq_domain)
    
    return np.array(tf_grid)

# 执行OFDM解调
Y_tf = ofdm_demodulate(rx_ofdm_symbols, M, cp_len)
print(f"OFDM解调完成")
print(f"时频域接收信号形状: {Y_tf.shape}")

In [ ]:
# ISFFT变换：时频域 → 延迟-多普勒域
Y_dd = sfft(Y_tf, N, M)
print(f"ISFFT变换完成")
print(f"延迟-多普勒域接收信号形状: {Y_dd.shape}")

In [ ]:
# 可视化：接收端信号在延迟-多普勒域
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 发送信号（延迟-多普勒域）
ax1 = axes[0, 0]
im1 = ax1.imshow(np.abs(X_dd), aspect='auto', origin='lower',
                  extent=[0, N, 0, M], cmap='Blues', interpolation='nearest')
plt.colorbar(im1, ax=ax1, label='|X(τ, ν)|')
ax1.set_xlabel('多普勒索引 (ν)', fontsize=11)
ax1.set_ylabel('延迟索引 (τ)', fontsize=11)
ax1.set_title('发送：延迟-多普勒域信号', fontsize=12)

# 信道（延迟-多普勒域）
ax2 = axes[0, 1]
im2 = ax2.imshow(np.abs(h_dd), aspect='auto', origin='lower',
                  extent=[-max_doppler, max_doppler, 0, max_delay+1],
                  cmap='hot', interpolation='nearest')
plt.colorbar(im2, ax=ax2, label='|h(τ, ν)|')
ax2.set_xlabel('多普勒索引 (ν)', fontsize=11)
ax2.set_ylabel('延迟索引 (τ)', fontsize=11)
ax2.set_title('信道：延迟-多普勒域冲激响应', fontsize=12)

# 接收信号（延迟-多普勒域）- 未均衡
ax3 = axes[1, 0]
im3 = ax3.imshow(np.abs(Y_dd), aspect='auto', origin='lower',
                  extent=[0, N, 0, M], cmap='Reds', interpolation='nearest')
plt.colorbar(im3, ax=ax3, label='|Y(τ, ν)|')
ax3.set_xlabel('多普勒索引 (ν)', fontsize=11)
ax3.set_ylabel('延迟索引 (τ)', fontsize=11)
ax3.set_title('接收：延迟-多普勒域信号（未均衡）', fontsize=12)

# 星座图对比
ax4 = axes[1, 1]
ax4.scatter(X_dd.flatten().real, X_dd.flatten().imag, c='blue', s=20, alpha=0.5,
           label='发送', edgecolors='black', linewidths=0.3)
ax4.scatter(Y_dd.flatten().real, Y_dd.flatten().imag, c='red', s=20, alpha=0.5,
           label='接收', edgecolors='black', linewidths=0.3)
ax4.axhline(y=0, color='k', linewidth=0.5)
ax4.axvline(x=0, color='k', linewidth=0.5)
ax4.set_xlabel('In-Phase (I)', fontsize=11)
ax4.set_ylabel('Quadrature (Q)', fontsize=11)
ax4.set_title('发送vs接收星座图（未均衡）', fontsize=12)
ax4.legend()
ax4.grid(True, alpha=0.3)
ax4.set_xlim(-1.5, 1.5)
ax4.set_ylim(-1.5, 1.5)
ax4.set_aspect('equal')

plt.tight_layout()
plt.show()

print("观察：信道的作用是简单的复数乘法！")
print("但由于信道未知，接收信号是扭曲的（相位旋转+幅度缩放）")
print("OTFS均衡：只需对每个格点除以信道响应即可")

In [ ]:
# 单抽头均衡（假设信道已知）
# 在延迟-多普勒域，信道作用是对每个格点的简单复数乘法
# 均衡：Y均衡 = Y / h

# 创建完整分辨率的信道矩阵（插值到N×M网格）
h_estimated = np.zeros((M, N), dtype=complex)
for delay_idx, doppler_idx, amp in channel_taps:
    if delay_idx < M and (doppler_idx + max_doppler) < N:
        h_estimated[delay_idx, doppler_idx + max_doppler] = amp

# 单抽头均衡
# 避免除以零
epsilon = 1e-6
H_inv = 1.0 / (h_estimated + epsilon)
X_dd_equalized = Y_dd * H_inv

print("单抽头均衡完成")
print("均衡方法：X_equalized = Y / h(τ, ν)")

In [ ]:
# 可视化：均衡前后的星座图
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 均衡前
ax1 = axes[0]
ax1.scatter(Y_dd.flatten().real, Y_dd.flatten().imag, c='red', s=20, alpha=0.5,
           edgecolors='black', linewidths=0.3)
ax1.axhline(y=0, color='k', linewidth=0.5)
ax1.axvline(x=0, color='k', linewidth=0.5)
ax1.set_xlabel('In-Phase (I)', fontsize=11)
ax1.set_ylabel('Quadrature (Q)', fontsize=11)
ax1.set_title('均衡前星座图', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(-1.5, 1.5)
ax1.set_ylim(-1.5, 1.5)
ax1.set_aspect('equal')

# 均衡后
ax2 = axes[1]
ax2.scatter(X_dd_equalized.flatten().real, X_dd_equalized.flatten().imag, 
           c='green', s=20, alpha=0.5, edgecolors='black', linewidths=0.3)
ax2.scatter(X_dd.flatten().real, X_dd.flatten().imag, c='blue', s=10, alpha=0.3,
           label='发送（参考）', edgecolors='black', linewidths=0.3)
ax2.axhline(y=0, color='k', linewidth=0.5)
ax2.axvline(x=0, color='k', linewidth=0.5)
ax2.set_xlabel('In-Phase (I)', fontsize=11)
ax2.set_ylabel('Quadrature (Q)', fontsize=11)
ax2.set_title('均衡后星座图', fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xlim(-1.5, 1.5)
ax2.set_ylim(-1.5, 1.5)
ax2.set_aspect('equal')

# 原始发送信号
ax3 = axes[2]
ax3.scatter(X_dd.flatten().real, X_dd.flatten().imag, c='blue', s=20, alpha=0.5,
           edgecolors='black', linewidths=0.3)
ax3.axhline(y=0, color='k', linewidth=0.5)
ax3.axvline(x=0, color='k', linewidth=0.5)
ax3.set_xlabel('In-Phase (I)', fontsize=11)
ax3.set_ylabel('Quadrature (Q)', fontsize=11)
ax3.set_title('发送星座图（原始）', fontsize=12)
ax3.grid(True, alpha=0.3)
ax3.set_xlim(-1.5, 1.5)
ax3.set_ylim(-1.5, 1.5)
ax3.set_aspect('equal')

plt.tight_layout()
plt.show()

print("观察：单抽头均衡后，星座图恢复到接近原始状态")
print("这就是OTFS的核心优势：延迟-多普勒域的均衡极其简单！")

In [ ]:
# QAM解映射
rx_qam_symbols = X_dd_equalized.flatten()
rx_bits = qam_demodulate(rx_qam_symbols, qam_constellation)

# 计算误码率
errors = np.sum(tx_bits != rx_bits)
ber = errors / len(tx_bits)

print(f"QAM解映射完成")
print(f"发射比特数: {len(tx_bits)}")
print(f"错误比特数: {errors}")
print(f"误码率 (BER): {ber:.6f}")

## 4. 时频域网格与延迟-多普勒域网格

### 4.1 两个域的对比

OTFS在两个域之间变换：

| 域 | 变量 | 网格表示 | 变换工具 |
|------|------|----------|----------|
| **时频域** | $(t, f)$ | $X(t, f)$ - 每个时间-频率格点放置子载波 | IFFT/FFT |
| **延迟-多普勒域** | $(\tau, \nu)$ | $X(\tau, \nu)$ - 每个延迟-多普勒格点放置QAM符号 | SFFT/ISFFT |

### 4.2 SFFT的物理意义

SFFT（对称有限傅里叶变换）是连接两个域的桥梁：

$$X(t, f) = \sum_{\tau} \sum_{\nu} X(\tau, \nu) e^{j2\pi \nu t} e^{j2\pi f \tau}$$

这意味着：
- 每个延迟-多普勒格点 $(\tau, \nu)$ 对应一个基函数 $e^{j2\pi \nu t} e^{j2\pi f \tau}$
- 时频域信号是这些基函数的叠加
- 基函数的能量在整个时频域扩散

In [ ]:
# 可视化：单点源在两个域的表示
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 创建单点信号（在延迟-多普勒域的一个格点上）
single_point_dd = np.zeros((M, N), dtype=complex)
single_point_dd[M//4, N//4] = 1.0 + 0j  # 延迟=M/4, 多普勒=N/4

# 可视化延迟-多普勒域
ax1 = axes[0, 0]
im1 = ax1.imshow(np.abs(single_point_dd), aspect='auto', origin='lower',
                  extent=[0, N, 0, M], cmap='Blues', interpolation='nearest')
plt.colorbar(im1, ax=ax1, label='|X(τ, ν)|')
ax1.scatter([N//4], [M//4], c='red', s=100, marker='x')
ax1.set_xlabel('多普勒索引 (ν)', fontsize=11)
ax1.set_ylabel('延迟索引 (τ)', fontsize=11)
ax1.set_title('延迟-多普勒域：单点信号', fontsize=12)

# SFFT变换后的时频域
single_point_tf = isfft(single_point_dd, N, M)
ax2 = axes[0, 1]
im2 = ax2.imshow(np.abs(single_point_tf), aspect='auto', origin='lower',
                  extent=[0, M, 0, N], cmap='Greens', interpolation='nearest')
plt.colorbar(im2, ax=ax2, label='|X(t, f)|')
ax2.set_xlabel('频率索引 (f)', fontsize=11)
ax2.set_ylabel('时间索引 (t)', fontsize=11)
ax2.set_title('时频域：单点信号SFFT后', fontsize=12)

# 时频域信号沿某时间切片的波形
ax3 = axes[0, 2]
time_slice = N // 2
ax3.plot(np.arange(M), np.abs(single_point_tf[time_slice, :]), 'g-o', markersize=4)
ax3.set_xlabel('频率索引', fontsize=11)
ax3.set_ylabel('幅度', fontsize=11)
ax3.set_title(f'时频域第{time_slice}行（时间切片）的频谱', fontsize=12)
ax3.grid(True, alpha=0.3)

# 另一个例子：沿延迟轴的信号
delay_slice_dd = np.zeros((M, N), dtype=complex)
delay_slice_dd[:, N//2] = 1.0  # 沿多普勒轴的一条线

ax4 = axes[1, 0]
im4 = ax4.imshow(np.abs(delay_slice_dd), aspect='auto', origin='lower',
                  extent=[0, N, 0, M], cmap='Blues', interpolation='nearest')
plt.colorbar(im4, ax=ax4, label='|X(τ, ν)|')
ax4.set_xlabel('多普勒索引 (ν)', fontsize=11)
ax4.set_ylabel('延迟索引 (τ)', fontsize=11)
ax4.set_title('延迟-多普勒域：多普勒切片', fontsize=12)

delay_slice_tf = isfft(delay_slice_dd, N, M)
ax5 = axes[1, 1]
im5 = ax5.imshow(np.abs(delay_slice_tf), aspect='auto', origin='lower',
                  extent=[0, M, 0, N], cmap='Greens', interpolation='nearest')
plt.colorbar(im5, ax=ax5, label='|X(t, f)|')
ax5.set_xlabel('频率索引 (f)', fontsize=11)
ax5.set_ylabel('时间索引 (t)', fontsize=11)
ax5.set_title('时频域：多普勒切片SFFT后', fontsize=12)

# 解释
ax6 = axes[1, 2]
ax6.axis('off')
explanation = """
关键洞察：

1. 单点信号在延迟-多普勒域
   → SFFT后能量扩散到整个时频域

2. 沿多普勒轴的切片
   → SFFT后变成沿时间轴的正弦波

3. 沿延迟轴的切片
   → SFFT后变成沿频率轴的正弦波

这解释了OTFS的工作原理：
- QAM符号在延迟-多普勒域是局部的、稀疏的
- SFFT将这些符号扩展到整个时频域
- 信道在延迟-多普勒域是稀疏的
- ISFFT将信道效应聚焦到局部
"""
ax6.text(0.1, 0.95, explanation, transform=ax6.transAxes, fontsize=10,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## 5. OTFS vs OFDM对比

### 5.1 系统框图对比

```
OFDM发送端：                     OTFS发送端：
QAM符号                           QAM符号
   ↓                                ↓
时频网格(X[k,l])              延迟-多普勒网格(X[τ,ν])
   ↓                                ↓
   ↓                           SFFT变换
   ↓                                ↓
IFFT                          时频网格(X[t,f])
   ↓                                ↓
添加CP                        IFFT + CP
   ↓                                ↓
时域发射                      时域发射

OFDM接收端：                     OTFS接收端：
时域接收                        时域接收
   ↓                                ↓
去除CP                         去除CP + FFT
   ↓                                ↓
FFT                           时频网格(Y[t,f])
   ↓                                ↓
   ↓                           ISFFT变换
   ↓                                ↓
均衡(每子载波)                 延迟-多普勒域(Y[τ,ν])
   ↓                                ↓
QAM解映射                     单抽头均衡
   ↓                                ↓
                              QAM解映射
```

In [ ]:
# 对比OTFS和OFDM在高移动性场景下的性能
def simulate_ofdm_vs_otfs(N, M, num_paths, max_doppler, snr_db=20):
    """
    对比OFDM和OTFS在高多普勒场景下的性能
    """
    np.random.seed(200)
    
    # 生成随机QAM数据
    qam_constellation, _ = create_qam_constellation(16)
    bits_per_symbol = 4
    
    # OFDM数据
    total_bits_ofdm = N * M * bits_per_symbol
    tx_bits_ofdm = np.random.randint(0, 2, total_bits_ofdm)
    qam_ofdm = qam_modulate(tx_bits_ofdm, qam_constellation)
    X_ofdm = qam_ofdm.reshape(N, M)  # shape (N_time, M_freq)
    
    # OTFS数据：同样的QAM符号放在延迟-多普勒域
    X_dd = qam_ofdm.reshape(M, N)  # shape (M_delay, N_doppler)
    
    # 创建多普勒信道（高多普勒）
    h_dd, channel_taps = create_delay_doppler_channel(num_paths, max_delay=5, 
                                                       max_doppler=max_doppler, N=N, M=M)
    
    # ========== OFDM仿真 ==========
    # OFDM调制
    cp_len = 8
    ofdm_symbols = ofdm_modulate(X_ofdm, M, cp_len)
    
    # 通过信道（简化：每条路径简单叠加）
    # 实际上OFDM在高多普勒下会有ICI（子载波间干扰）
    rx_ofdm_symbols = np.zeros_like(ofdm_symbols, dtype=complex)
    for delay_idx, doppler_idx, amp in channel_taps:
        for sym_idx in range(N):
            doppler_phase = np.exp(1j * 2 * np.pi * doppler_idx * sym_idx / N)
            gain = amp * doppler_phase
            start = cp_len + delay_idx
            end = start + M
            if end <= ofdm_symbols.shape[1]:
                rx_ofdm_symbols[sym_idx, start:end] += ofdm_symbols[sym_idx, cp_len:cp_len+M] * gain
    
    # 添加噪声
    signal_power = np.mean(np.abs(rx_ofdm_symbols)**2)
    noise_power = signal_power / (10**(snr_db/10))
    rx_ofdm_symbols += np.sqrt(noise_power/2) * (np.random.randn(*rx_ofdm_symbols.shape) + 
                                                  1j * np.random.randn(*rx_ofdm_symbols.shape))
    
    # OFDM解调
    Y_ofdm_tf = ofdm_demodulate(rx_ofdm_symbols, M, cp_len)
    
    # OFDM均衡（每子载波单抽头，简化）
    H_ofdm = np.zeros((N, M), dtype=complex)
    for delay_idx, doppler_idx, amp in channel_taps:
        if delay_idx < M and doppler_idx + max_doppler < N:
            H_ofdm[:, doppler_idx + max_doppler] += amp
    
    Y_ofdm_equalized = Y_ofdm_tf / (H_ofdm + 1e-6)
    
    # OFDM误码
    rx_bits_ofdm = qam_demodulate(Y_ofdm_equalized.flatten(), qam_constellation)
    errors_ofdm = np.sum(tx_bits_ofdm[:len(rx_bits_ofdm)] != rx_bits_ofdm)
    ber_ofdm = errors_ofdm / len(rx_bits_ofdm)
    
    # ========== OTFS仿真 ==========
    # OTFS调制
    X_tf = isfft(X_dd, N, M)
    otfs_symbols = ofdm_modulate(X_tf, M, cp_len)
    
    # 通过同一信道
    rx_otfs_symbols = np.zeros_like(otfs_symbols, dtype=complex)
    for delay_idx, doppler_idx, amp in channel_taps:
        for sym_idx in range(N):
            doppler_phase = np.exp(1j * 2 * np.pi * doppler_idx * sym_idx / N)
            gain = amp * doppler_phase
            start = cp_len + delay_idx
            end = start + M
            if end <= otfs_symbols.shape[1]:
                rx_otfs_symbols[sym_idx, start:end] += otfs_symbols[sym_idx, cp_len:cp_len+M] * gain
    
    # 添加相同噪声
    rx_otfs_symbols += np.sqrt(noise_power/2) * (np.random.randn(*rx_otfs_symbols.shape) + 
                                                  1j * np.random.randn(*rx_otfs_symbols.shape))
    
    # OTFS解调
    Y_otfs_tf = ofdm_demodulate(rx_otfs_symbols, M, cp_len)
    Y_otfs_dd = sfft(Y_otfs_tf, N, M)
    
    # OTFS均衡（单抽头）
    h_otfs = np.zeros((M, N), dtype=complex)
    for delay_idx, doppler_idx, amp in channel_taps:
        if delay_idx < M and doppler_idx + max_doppler < N:
            h_otfs[delay_idx, doppler_idx + max_doppler] = amp
    
    Y_otfs_equalized = Y_otfs_dd / (h_otfs + 1e-6)
    
    # OTFS误码
    rx_bits_otfs = qam_demodulate(Y_otfs_equalized.flatten(), qam_constellation)
    errors_otfs = np.sum(tx_bits_ofdm[:len(rx_bits_otfs)] != rx_bits_otfs)
    ber_otfs = errors_otfs / len(rx_bits_otfs)
    
    return ber_ofdm, ber_otfs

# 测试不同多普勒下的性能
doppler_values = [1, 3, 5, 7, 10]
ber_ofdm_list = []
ber_otfs_list = []

for doppler in doppler_values:
    ber_ofdm, ber_otfs = simulate_ofdm_vs_otfs(N=32, M=32, num_paths=3, 
                                               max_doppler=doppler, snr_db=20)
    ber_ofdm_list.append(ber_ofdm)
    ber_otfs_list.append(ber_otfs)
    print(f"多普勒={doppler}: OFDM BER={ber_ofdm:.6f}, OTFS BER={ber_otfs:.6f}")

In [ ]:
# 可视化：OTFS vs OFDM性能对比
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 误码率对比
ax1 = axes[0]
ax1.semilogy(doppler_values, ber_ofdm_list, 'r-o', linewidth=2, markersize=8, label='OFDM')
ax1.semilogy(doppler_values, ber_otfs_list, 'b-s', linewidth=2, markersize=8, label='OTFS')
ax1.set_xlabel('最大多普勒偏移', fontsize=11)
ax1.set_ylabel('误码率 (BER)', fontsize=11)
ax1.set_title('OTFS vs OFDM：高移动性场景性能对比', fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)

# 星座图对比（高多普勒场景）
ber_ofdm, ber_otfs = simulate_ofdm_vs_otfs(N=32, M=32, num_paths=3, 
                                          max_doppler=10, snr_db=20)

# 重新运行获取星座数据
np.random.seed(200)
qam_constellation, _ = create_qam_constellation(16)
bits_per_symbol = 4
total_bits = N * M * bits_per_symbol
tx_bits = np.random.randint(0, 2, total_bits)
qam_data = qam_modulate(tx_bits, qam_constellation)

# OTFS星座
X_dd = qam_data.reshape(M, N)
X_tf = isfft(X_dd, N, M)
otfs_syms = ofdm_modulate(X_tf, M, cp_len)

ax2 = axes[1]
ax2.scatter(qam_data.real, qam_data.imag, c='blue', s=10, alpha=0.3, 
           label='理想星座', edgecolors='none')

# 添加信道后的接收星座（示意）
h_dd, channel_taps = create_delay_doppler_channel(3, 5, 10, N, M)
rx_otfs = np.zeros_like(otfs_syms, dtype=complex)
for delay_idx, doppler_idx, amp in channel_taps:
    for sym_idx in range(N):
        doppler_phase = np.exp(1j * 2 * np.pi * doppler_idx * sym_idx / N)
        gain = amp * doppler_phase
        start = cp_len + delay_idx
        end = start + M
        if end <= otfs_syms.shape[1]:
            rx_otfs[sym_idx, start:end] += otfs_syms[sym_idx, cp_len:cp_len+M] * gain

Y_tf = ofdm_demodulate(rx_otfs, M, cp_len)
Y_dd = sfft(Y_tf, N, M)
h_full = np.zeros((M, N), dtype=complex)
for delay_idx, doppler_idx, amp in channel_taps:
    if delay_idx < M and doppler_idx + 10 < N:
        h_full[delay_idx, doppler_idx + 10] = amp
Y_eq = Y_dd / (h_full + 1e-6)

ax2.scatter(Y_eq.flatten().real, Y_eq.flatten().imag, c='red', s=10, alpha=0.5, 
           label='OTFS均衡后', edgecolors='none')
ax2.axhline(y=0, color='k', linewidth=0.5)
ax2.axvline(x=0, color='k', linewidth=0.5)
ax2.set_xlabel('In-Phase (I)', fontsize=11)
ax2.set_ylabel('Quadrature (Q)', fontsize=11)
ax2.set_title('OTFS均衡后星座图', fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xlim(-1.5, 1.5)
ax2.set_ylim(-1.5, 1.5)
ax2.set_aspect('equal')

plt.tight_layout()
plt.show()

print("观察：")
print("1. 高多普勒场景下，OTFS比OFDM具有更好的误码性能")
print("2. OTFS将多普勒作为网格维度处理，而非干扰源")
print("3. 在延迟-多普勒域，所有QAM符号经历相同的信道响应")

## 6. 关联OTFS论文

### 6.1 对应论文章节

本notebook的内容与以下论文章节对应：

**OTFS论文："Orthogonal Time Frequency Space Modulation" (作者：Hadani等, 2017)**

| Notebook章节 | 论文章节 | 内容 |
|-------------|----------|------|
| OTFS调制原理 | 第2节 | OTFS调制框架、时频域与延迟-多普勒域变换 |
| 代码演示 | 第2.4-2.7节 | SFFT/ISFFT变换、OFDM兼容实现 |
| 时频域与延迟-多普勒域 | 第3节 | 网格表示、基函数定义 |
| OTFS vs OFDM | 第4节 | 性能对比、优势分析 |

### 6.2 论文核心要点

**第2.4节 - OTFS调制框架**：
OTFS通过Zak变换（SFFT）将延迟-多普勒域信号变换到时频域，再通过OFDM调制发射。

**第2.5节 - 与OFDM的关系**：
OTFS可以看作OFDM的泛化形式。OFDM是OTFS在特定参数选择下的特例。

**第3节 - 信道表示与均衡**：
在延迟-多普勒域，信道表示为稀疏的冲激响应 $h(\tau, \nu)$，均衡简化为单抽头复数乘法。

## 7. 思考题

### 思考题 1
解释为什么在延迟-多普勒域中信道是"时不变"的。这相对于OFDM的"时变"信道有什么优势？

### 思考题 2
OTFS使用SFFT（对称有限傅里叶变换）代替OFDM中的IFFT/FFT。请解释：
- SFFT = FFT行 + IFFT列
- 为什么这个组合能实现Zak变换？
- 逆变换需要什么操作？

### 思考题 3
假设某OTFS系统使用N=128（时间维度）和M=512（频率维度）的网格。
- 该系统能承载多少QAM符号？
- 如果使用16-QAM，每个OTFS帧能传输多少比特？

### 思考题 4
高速铁路场景（速度350 km/h，载波频率2.6 GHz）：
- 计算最大多普勒频移
- 如果使用OFDM（子载波间隔15kHz），多普勒扩展会如何影响系统？
- OTFS如何处理这种高移动性场景？

### 思考题 5
在延迟-多普勒域，信道通常是稀疏的（只有几条主要路径）。
- 这种稀疏性对信道估计有什么影响？
- 如何利用压缩感知技术来减少导频开销？

### 思考题 6
比较OFDM和OTFS的接收机复杂度：
- OFDM需要什么均衡操作？
- OTFS需要什么均衡操作？
- 为什么OTFS的均衡更简单？

### 思考题 7（扩展）
设计一个完整的OTFS系统仿真，验证以下流程：
1. 生成随机QAM数据
2. 放置到延迟-多普勒网格
3. SFFT变换到时频域
4. OFDM调制（含CP）
5. 通过多径多普勒信道
6. OFDM解调
7. ISFFT变换回延迟-多普勒域
8. 单抽头均衡
9. QAM解映射
10. 计算BER并与OFDM对比

---

**参考答案提示**

**思考题3**：N×M = 128×512 = 65536个QAM符号，16-QAM每个符号4比特，总共262144比特

**思考题4**：多普勒频移 ≈ 2.6GHz × (350/108) / 3e8 ≈ 756 Hz。OFDM中这会导致ICI，但OTFS将其作为网格维度处理

---

## 总结

本notebook介绍了OTFS调制解调的全流程：

1. **OTFS核心原理**：将QAM符号放置在延迟-多普勒域，实现时不变信道

2. **调制流程**：QAM符号 → 延迟-多普勒网格 → SFFT → 时频域 → IFFT+CP → 时域

3. **解调流程**：时域 → 去除CP+FFT → 时频域 → ISFFT → 延迟-多普勒域 → 单抽头均衡 → QAM解映射

4. **关键优势**：
   - 延迟-多普勒域中信道时不变
   - 单抽头均衡（简单复数乘法）
   - 高移动性场景下性能优于OFDM
   - 天然利用信道稀疏性

5. **与前面notebook的关联**：
   - Fourier变换（信号表示基础）
   - 延迟-多普勒域（信道表示）
   - Zak变换/SFFT（OTFS核心数学工具）
   - OFDM（OTFS的调制基础）

OTFS代表了通信调制技术的重要发展方向，特别是在高移动性、高频率选择性和稀疏信道场景下具有显著优势。